In [9]:
pip install mp-api

In [2]:
import pandas as pd
from mp_api.client import MPRester
API_KEY = "API_KEY" # Add API_KEY from Materials Project Website -> It's different for every user and MP recommends its not shared
mpr = MPRester(API_KEY)

In [3]:
docs = mpr.materials.insertion_electrodes.search(
        working_ion=["Li","Na","Mg","Ca","Ag"], # We can change this to find data for other types of batteries using working ions like Na, Ca ions etc.
        fields=[
            "battery_id",
            "formula_charge",
            "formula_discharge",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "stability_charge",
            "stability_discharge",
            "battery_type",
            "battery_formula",
            "elements",
            "host_structure",
            "max_delta_volume",
            "working_ion",
            "id_charge",
            "id_discharge",
            "nelements",
         ] # There are a lot more properties available, but not all of them seem to be useful and to save space, it's better to choose the necessary ones.
       )

data = [doc.dict() for doc in docs]
df = pd.DataFrame(data)

#df.to_csv("mp_battery_data.csv", index=False) # Save data to a csv file


Retrieving InsertionElectrodeDoc documents:   0%|          | 0/5783 [00:00<?, ?it/s]

In [4]:
print(df.head())

  battery_type    battery_id battery_formula working_ion elements  nelements  \
0    insertion      mp-28_Li         Li0-3Ce          Li     [Ce]          1   
1    insertion    mp-2074_Li         Li0-3Sb          Li     [Sb]          1   
2    insertion  mp-568806_Li       Li0-0.17C          Li      [C]          1   
3    insertion  mp-573471_Li    Li4.25-4.4Sn          Li     [Sn]          1   
4    insertion   mp-22902_Li         Li0-1Bi          Li     [Bi]          1   

  formula_charge formula_discharge  max_delta_volume  average_voltage  \
0             Ce             Li3Ce          2.951183        -0.444846   
1             Sb             Li3Sb          1.569237         1.015953   
2              C              LiC6          0.025414         0.084405   
3        Li17Sn4           Li22Sn5          0.009450        -0.362516   
4             Bi              LiBi          0.368255         0.796796   

   capacity_grav  energy_grav  stability_charge  stability_discharge  \
0     49

In [6]:
discharge_ids = df['id_discharge'].unique().tolist()

extra_docs = mpr.materials.summary.search(
    material_ids=discharge_ids,
    fields=[
        "material_id",
        "band_gap",
        "formation_energy_per_atom", # This is your energy of formation
        "density",                   # Physical density
        "structure"             # Cubic, Monoclinic, etc.
    ]
)

extra_df = pd.DataFrame([doc.dict() for doc in extra_docs])
df = df.merge(extra_df, left_on='id_discharge', right_on='material_id', how='left')

Retrieving SummaryDoc documents:   0%|          | 0/5773 [00:00<?, ?it/s]

In [7]:
print(df.head())

  battery_type    battery_id battery_formula working_ion elements  nelements  \
0    insertion      mp-28_Li         Li0-3Ce          Li     [Ce]          1   
1    insertion    mp-2074_Li         Li0-3Sb          Li     [Sb]          1   
2    insertion  mp-568806_Li       Li0-0.17C          Li      [C]          1   
3    insertion  mp-573471_Li    Li4.25-4.4Sn          Li     [Sn]          1   
4    insertion   mp-22902_Li         Li0-1Bi          Li     [Bi]          1   

  formula_charge formula_discharge  max_delta_volume  average_voltage  ...  \
0             Ce             Li3Ce          2.951183        -0.444846  ...   
1             Sb             Li3Sb          1.569237         1.015953  ...   
2              C              LiC6          0.025414         0.084405  ...   
3        Li17Sn4           Li22Sn5          0.009450        -0.362516  ...   
4             Bi              LiBi          0.368255         0.796796  ...   

   id_charge  id_discharge                        

In [8]:
#df.to_csv("mp_battery_data.csv", index=False) # Save data to a csv file